# Tarea 16 - Enfriamiento termico de la corteza de una estrella de neutrones

**Curso:** Astrofisica Computacional 2026-I  
**Metodo:** Diferencias finitas explicitas (FTCS)  

---

Se modela la evolucion temporal de la temperatura $T(z,t)$ en la corteza de una estrella de neutrones con la ecuacion

$$
\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial z^2} - \epsilon_\nu T^5,
$$

donde el primer termino representa la difusion termica y el segundo la perdida de energia por emision de neutrinos.

Se imponen condiciones de frontera de Dirichlet:

$$
T(0,t)=T_{\rm superficie}, \qquad T(L,t)=T_{\rm interior}.
$$


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})


## Parametros fisicos y numericos

Tomamos una malla uniforme en profundidad $z \in [0,L]$ y avanzamos en el tiempo con el esquema FTCS.
Para estabilidad del metodo explicito se usa la condicion CFL:

$$
\Delta t \leq \frac{\Delta z^2}{2\alpha}.
$$

Se adopta un factor de seguridad de $0.4$.


In [ ]:
# ============================================================
# Parametros fisicos
# ============================================================
L = 1000.0          # profundidad de la corteza [m]
alpha = 1.0         # difusividad termica
eps_nu = 1e-45      # constante de emision de neutrinos
T_inicial = 1e9     # temperatura inicial uniforme [K]

# Condiciones de frontera
T_superficie = 1e6  # temperatura en z = 0 [K]
T_interior = 1e9    # temperatura en z = L [K]

# ============================================================
# Parametros numericos
# ============================================================
Nz = 100
dz = L / (Nz - 1)
dt = 0.4 * dz**2 / alpha
Nt = 5000

print(f"Nz = {Nz}")
print(f"dz = {dz:.4f} m")
print(f"dt = {dt:.4f} s")
print(f"Tiempo total simulado = {Nt * dt:.4f} s")


## Implementacion del metodo FTCS

La discretizacion para los nodos interiores queda:

$$
T_i^{n+1} = T_i^n + \Delta t \left[ \alpha \frac{T_{i+1}^n - 2T_i^n + T_{i-1}^n}{\Delta z^2} - \epsilon_\nu (T_i^n)^5 \right].
$$

En cada paso se conservan fijas las temperaturas de la superficie y del interior.


In [ ]:
z = np.linspace(0.0, L, Nz)
T = np.full(Nz, T_inicial, dtype=float)
T[0] = T_superficie
T[-1] = T_interior

perfiles = {0: T.copy()}
pasos_para_graficar = [100, 1000, 2500, 4999]

for n in range(Nt):
    T_nueva = T.copy()

    difusion = alpha * (T[2:] - 2.0 * T[1:-1] + T[:-2]) / dz**2
    sumidero = eps_nu * T[1:-1]**5
    T_nueva[1:-1] = T[1:-1] + dt * (difusion - sumidero)

    T_nueva[0] = T_superficie
    T_nueva[-1] = T_interior
    T = T_nueva

    if n in pasos_para_graficar:
        perfiles[n] = T.copy()

for paso in [0] + pasos_para_graficar:
    tiempo = paso * dt
    print(f"Paso {paso:4d} -> t = {tiempo:10.4f} s, T_min = {perfiles[paso].min():.3e} K, T_max = {perfiles[paso].max():.3e} K")


## Grafica del perfil de temperatura

Se comparan el perfil inicial y varios instantes de la evolucion temporal para observar el enfriamiento de la corteza cerca de la superficie.


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(z, perfiles[0], linestyle='--', linewidth=2.0, label='t = 0 s')

for paso in pasos_para_graficar:
    tiempo_actual = paso * dt
    plt.plot(z, perfiles[paso], linewidth=2.0, label=f'Paso {paso} (t ~ {tiempo_actual:.2f} s)')

plt.title('Enfriamiento termico de la corteza de una estrella de neutrones')
plt.xlabel('Profundidad z [m]')
plt.ylabel('Temperatura T [K]')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()
